<a href="https://colab.research.google.com/github/ze-16/Speech-training-agent/blob/main/IEMOCAP_processing_script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **For compliance reasons, this script only shows the process of extracting the files from the corpus and does not contain access to the corpus**

In [ ]:
import pandas as pd
import numpy as np
import gdown
import os
import tarfile

In [ ]:
# Final dataset function definition to merge audio ID, labels and sentences.
def create_text_csv(wav,labels,sentences,path):
  df = []

  for audio_id in wav:

    row = {
        'Recording_name': audio_id,
        'Emotion': labels[audio_id],
        'Sentence': sentences[audio_id]
    }
    df.append(row)
    # indentation error caused incorrect csv file writing

  final_df = pd.DataFrame(df)
  final_df.to_csv(path,index=False)

In [ ]:
# Path for Tarfile library to access later on, relative local path not public link code still ethicaly compliant
tar_path = "/content/drive/MyDrive/YOUR_FOLDER_HERE/IEMOCAP_full_release_withoutVideos (1).tar.gz"
wav_output = "/content/training_data/wavs"
Label_sentence_data = "/content/training_data/label_sentence_data.csv"

In [ ]:
# Making a path in the directory
os.makedirs(wav_output,exist_ok=True)

In [ ]:
# Collection of dictonaries and lists to be used later
labels_dic = {}
sentences_dic = {}
wav_ids = []

In [ ]:
# Emotion mapping to be used for the IEMOCAP corpus
emo_map = {
    'neu': 0,
    'exc': 1,
    'hap': 1,
    'sad': 2,
    'fru': 3,
    'dis': 3,
    'ang': 3
}

In [ ]:
# Code block to process iemocap corpus and produce dictionaries/lists with emotion labels, sentence and IDs
# First loop, to create a dictionary with the emotion labels mapped to the label IDs
with tarfile.open(tar_path) as t:
  for member in t.getmembers():
    if "/dialog/EmoEvaluation" in member.name and member.name.endswith(".txt"):
      f = t.extractfile(member)
      if f is not None:
        content = f.read().decode('utf-8')
        for l in content.split('\n'):
          parts = l.split('\t')
          if len(parts) >= 3 and parts[1].startswith('Ses'):
            wav_id = parts[1]
            emotion_label = parts[2]
            labels_dic[wav_id] = emo_map.get(emotion_label, 99)

# Second loop, to create a dictionary of the sentences
    elif "dialog/transcriptions" in member.name and member.name.endswith(".txt"):
      f = t.extractfile(member)
      if f is not None:
        content = f.read().decode('utf-8')
        for l in content.split('\n'):
          parts = l.split(']:')
          wav_id = parts[0].split ('[')[0].strip()
          if wav_id.startswith('Ses'):
            sentence = parts[1].strip()
            sentences_dic[wav_id] = sentence

In [ ]:
# Removes default emotion label 99 which represents error in emotion calssification
keys_to_remove = []
for lbl,lbl_emo in labels_dic.items():
  if lbl_emo == 99:
    keys_to_remove.append(lbl)
for key in keys_to_remove:
  labels_dic.pop(key)

In [1]:
# print(labels_dic), code commented out
# Extra cautionary step to adhere to data compliance

In [ ]:
# Loop checks if the same recording ID exists in both dictionaries and adds the ID to a list
for i in labels_dic.keys():
  if i in sentences_dic.keys():
    wav_ids.append(i)

In [ ]:
# Create a csv with the recording IDs used as a reference
create_text_csv(wav_ids,labels_dic,sentences_dic,Label_sentence_data)

In [ ]:
# Final tar-loop, to fetch the recordings based on recording IDs
with tarfile.open(tar_path) as t:
  for w in t.getmembers():
    if "sentences/wav" in w.name and w.name.endswith(".wav"):
      w_id = w.name.split('/')[-1].split('.wav')[0]
      if w_id in wav_ids:
        f = t.extractfile(w)
        if f is not None:
          wav_data = f.read()
          with open(os.path.join(wav_output,w_id+'.wav'),'wb') as wav_file:
            wav_file.write(wav_data)


 Use this code to save iemocap extracted data to your drive, remove the comment symbols and add the path in your drive you want the extracted files to be saved to.

In [ ]:
# import shutil

# shutil.copytree('','',)